# Real Data Resolution Sensitivity — 2024-07-03

**Goal**: Fit HybridVecchia on real GEMS TCO data at 4 observation densities
(x8=coarsest → x1=finest, via maxmin-order thinning) × 2 smooth values.
Track how parameter estimates change as data volume increases.

**Design**: 4 resolutions × 2 smooth = 8 fits

| fit_smooth | R=8 | R=4 | R=2 | R=1 |
|---|---|---|---|---|
| 0.5 | ✓ | ✓ | ✓ | ✓ |
| 1.5 | ✓ | ✓ | ✓ | ✓ |

**Hybrid Vecchia conditioning sets** (same as simulation notebook):
- Set A (t):   spatial NN 20
- Set B (t-1): anchor(1) + local NN(16) + upstream shifted NN(2)
- Set C (t-2): anchor(1) + local NN(12) + upstream shifted×2 NN(2)

**Thinning**: after MaxMin ordering, take every Rth row.
R=8 ≈ 2k obs total; R=1 ≈ 140k obs total.

In [ ]:
import sys, time, gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

# Absolute path — CWD-independent (mirrors what data_loader.py itself uses)
sys.path.insert(0, '/Users/joonwonlee/Documents/GEMS_TCO-1/src')

from GEMS_TCO import configuration as config
from GEMS_TCO import orderings as _orderings
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_vecchia_hybrid import HybridVecchiaFit

print('torch:', torch.__version__)
print('device: cpu (Mac)')

In [ ]:
# ── experiment config ──────────────────────────────────────────────────
DEVICE = torch.device('cpu')
DTYPE  = torch.float64

YEAR      = '2024'
MONTH     = 7
DAY_IDX   = 2          # 0-based → July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]

DATE_STR  = f"{YEAR}{MONTH:02d}{DAY_IDX+1:02d}"   # e.g. 20240703

# ── Vecchia hyperparams ────────────────────────────────────────────────
MM_COND_NUMBER  = 100
NHEADS          = 100
LIMIT_A         = 20
LIMIT_B_LOCAL   = 16
LIMIT_C_LOCAL   = 12
DAILY_STRIDE    = 2
LAG1_FRESH      = 2
LAG2_FRESH      = 2
LAG1_LON_OFFSET = 0.063  # fixed structural hyperparameter (same as simulation)
                          # lag2 center = 2 × 0.063 (handled by multiplier=2.0 in _build_shift_lookup)

# ── optimizer ─────────────────────────────────────────────────────────
LBFGS_LR    = 1.0
LBFGS_STEPS = 10
LBFGS_EVAL  = 20
LBFGS_HIST  = 10

# ── experiment axes ───────────────────────────────────────────────────
FIT_SMOOTHS      = [0.5, 1.5]
RESOLUTION_MULTS = [8, 4, 2, 1]   # thinning factor; 8=coarsest, 1=finest

# ── initial values (from reference fit file) ──────────────────────────
INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon':  -0.1689,
    'nugget':     0.247,
}

P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time',
            'advec_lat', 'advec_lon', 'nugget']
P_DISP   = [r'$\sigma^2$ (sigmasq)', 'range_lat', 'range_lon', 'range_time',
            r'$\alpha_{lat}$', r'$\alpha_{lon}$', 'nugget']

print(f'Config loaded. Date: {DATE_STR}')

In [ ]:
# ── parameter reparametrization (same as simulation notebook) ──────────
def phys_to_log(d):
    """Physical params → log-reparametrized [logφ1, logφ2, logφ3, logφ4, advec_lat, advec_lon, log_nugget]."""
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4),
            d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    """Log-reparametrized → physical."""
    p    = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq':    np.exp(p[0]) / phi2,
        'range_lat':  rlon / phi3 ** 0.5,
        'range_lon':  rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat':  p[4],
        'advec_lon':  p[5],
        'nugget':     np.exp(p[6]),
    }


INIT_LOG = phys_to_log(INIT_DICT)
print('Initial log-params:', [f'{v:.4f}' for v in INIT_LOG])

In [ ]:
# ── load full data (finest resolution) ───────────────────────────────
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

# July 3 → day_idx=2 → hour indices [16, 24]
hour_indices = [DAY_IDX * 8, (DAY_IDX + 1) * 8]
full_day_map, _ = loader.load_working_data(
    df_map, monthly_mean, hour_indices,
    ord_mm=ord_mm, dtype=DTYPE, keep_ori=True,
)

sorted_df_keys = sorted(df_map.keys())
day_df_keys    = sorted_df_keys[hour_indices[0]:hour_indices[1]]
print(f'Day keys: {day_df_keys[0]}  ...  {day_df_keys[-1]}')

n_grid_full = next(iter(full_day_map.values())).shape[0]
n_valid_full = sum(int((~torch.isnan(v[:, 2])).sum()) for v in full_day_map.values())
print(f'Full grid: {n_grid_full} cells | valid obs across 8 time steps: {n_valid_full:,}')
print(f'Monthly mean TCO: {monthly_mean:.4f}')

In [ ]:
# ── extract grid coordinates for shift lookup ─────────────────────────
# Use Latitude/Longitude (grid centers, always valid) in maxmin order.
# Source_Lat/Lon (col 0-1 in tensor) can be NaN when no obs at a cell.
first_df = df_map[day_df_keys[0]].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)
# grid_coords_ordered: shape (n_grid, 2) with (lat, lon) — always finite
print(f'grid_coords_ordered shape: {grid_coords_ordered.shape}')
print(f'lat range: {grid_coords_ordered[:,0].min():.2f} – {grid_coords_ordered[:,0].max():.2f}')
print(f'lon range: {grid_coords_ordered[:,1].min():.2f} – {grid_coords_ordered[:,1].max():.2f}')

In [ ]:
# ── build thinned setups for each resolution ──────────────────────────
# Thinning: take every Rth row from MaxMin-ordered tensors.
# MaxMin order ensures the first rows are the most spatially spread-out,
# so every-Rth-row gives a spatially representative subsample.

setups = {}

for R in RESOLUTION_MULTS:
    thin_idx = np.arange(0, n_grid_full, R)

    # Thin tensors (each time step)
    thin_map = {k: v[thin_idx] for k, v in full_day_map.items()}

    # Grid coords for thinned set
    thin_spatial_coords = grid_coords_ordered[thin_idx]          # (lat, lon) for shift lookup
    thin_coords_lonlat  = thin_spatial_coords[:, ::-1].copy()    # (lon, lat) for find_nns_l2

    # Recompute NNS map on thinned spatial coords
    nns_thin = _orderings.find_nns_l2(
        locs=thin_coords_lonlat,
        max_nn=MM_COND_NUMBER,
    )

    n_valid = sum(int((~torch.isnan(v[:, 2])).sum()) for v in thin_map.values())

    setups[R] = {
        'thin_map':       thin_map,
        'nns':            nns_thin,
        'spatial_coords': thin_spatial_coords,
        'n_valid':        n_valid,
        'n_grid':         len(thin_idx),
    }
    print(f'R={R:2d}: n_grid={len(thin_idx):6,}  valid_obs={n_valid:8,}  nns_shape={nns_thin.shape}')

In [ ]:
# ── fitting loop: 4 resolutions × 2 smooth = 8 fits ──────────────────
results = []

for fs in FIT_SMOOTHS:
    for R in RESOLUTION_MULTS:
        setup = setups[R]
        tag   = f'smooth={fs}  R={R}  [n_obs={setup["n_valid"]:,}]'
        print(f"\n{'='*65}")
        print(f'Fitting: {tag}')
        print(f"  n_grid={setup['n_grid']}  n_valid={setup['n_valid']:,}")

        params = [
            torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True)
            for v in INIT_LOG
        ]

        model = HybridVecchiaFit(
            smooth=fs,
            input_map={k: v.to(DEVICE) for k, v in setup['thin_map'].items()},
            nns_map=setup['nns'],
            mm_cond_number=MM_COND_NUMBER,
            nheads=NHEADS,
            limit_A=LIMIT_A,
            limit_B_local=LIMIT_B_LOCAL,
            limit_C_local=LIMIT_C_LOCAL,
            daily_stride=DAILY_STRIDE,
            spatial_coords=setup['spatial_coords'],
            lag1_lon_offset=LAG1_LON_OFFSET,
            lag1_fresh_count=LAG1_FRESH,
            lag2_fresh_count=LAG2_FRESH,
        )
        model.precompute_conditioning_sets()

        optimizer = model.set_optimizer(
            params, lr=LBFGS_LR,
            max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL,
            history_size=LBFGS_HIST,
        )
        t0 = time.time()
        out, fit_iter = model.fit_vecc_lbfgs(
            params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5,
        )
        elapsed = time.time() - t0

        loss = float(out[-1])
        est  = backmap_params(out)
        print(f'  loss={loss:.4f}  elapsed={elapsed:.1f}s  steps={fit_iter+1}')
        for k in P_LABELS:
            print(f'    {k:12s}: {est[k]:10.4f}')

        results.append({
            'fit_smooth':  fs,
            'R':           R,
            'n_obs_total': setup['n_valid'],
            'n_grid':      setup['n_grid'],
            'loss':        loss,
            'fit_steps':   fit_iter + 1,
            'elapsed_s':   round(elapsed, 1),
            **{f'est_{k}': est[k] for k in P_LABELS},
        })
        del model
        gc.collect()

df = pd.DataFrame(results)
print(f'\nDone. {len(df)} fits recorded.')
print(df[['fit_smooth', 'R', 'n_obs_total', 'loss', 'fit_steps', 'elapsed_s']].to_string(index=False))

In [ ]:
# ── parameter trajectory plots ────────────────────────────────────────
# Color:  fit_smooth (blue=0.5, red=1.5)
# Marker: circle=0.5, square=1.5

COLOR  = {0.5: 'steelblue', 1.5: 'tomato'}
MK     = {0.5: 'o',         1.5: 's'}
SMOOTH_LABEL = {0.5: 'Matérn 1/2 (smooth=0.5)', 1.5: 'Matérn 3/2 (smooth=1.5)'}

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (param, plabel) in enumerate(zip(P_LABELS, P_DISP)):
    ax   = axes[i]
    init = INIT_DICT[param]

    for fs in FIT_SMOOTHS:
        sub = df[df.fit_smooth == fs].sort_values('n_obs_total')
        ax.plot(
            sub['n_obs_total'], sub[f'est_{param}'],
            color=COLOR[fs], marker=MK[fs], ms=8, lw=2,
            label=SMOOTH_LABEL[fs],
        )
        for _, row in sub.iterrows():
            ax.annotate(
                f"x{int(row.R)}",
                (row['n_obs_total'], row[f'est_{param}']),
                textcoords='offset points', xytext=(4, 4),
                fontsize=7, color='grey',
            )

    ax.axhline(init, color='grey', ls=':', lw=1.2, label=f'init={init:.3g}')
    ax.set_title(plabel, fontsize=13)
    ax.set_xlabel('Valid obs count  (x8=coarsest → x1=finest)')
    ax.set_ylabel('Estimate')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.25)

axes[-1].set_visible(False)

legend_entries = [
    mlines.Line2D([], [], color=COLOR[0.5], marker='o', lw=2, label='fit smooth=0.5 (Matérn 1/2)'),
    mlines.Line2D([], [], color=COLOR[1.5], marker='s', lw=2, label='fit smooth=1.5 (Matérn 3/2)'),
    mlines.Line2D([], [], color='grey',    ls=':',     lw=1.5, label='initial value'),
]
axes[-1].legend(handles=legend_entries, loc='center', fontsize=10, frameon=True)
axes[-1].set_visible(True)
axes[-1].axis('off')

fig.suptitle(
    f'Real Data — {DATE_STR[:4]}-{DATE_STR[4:6]}-{DATE_STR[6:]}  |  lat={LAT_RANGE}°  lon={LON_RANGE}°\n'
    'Parameter Estimates vs Observation Count  (x8=coarsest → x1=finest)',
    fontsize=13, y=1.01,
)
plt.tight_layout()
fname = f'real_data_param_trajectories_{DATE_STR}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

In [ ]:
# ── numerical summary ─────────────────────────────────────────────────
print('=' * 75)
print(f'REAL DATA RESOLUTION SENSITIVITY — 2024-07-03')
print('=' * 75)

for fs in FIT_SMOOTHS:
    sub = df[df.fit_smooth == fs].sort_values('n_obs_total')
    print(f'\nfit_smooth={fs}')
    print(f"  {'n_obs':>8}  {'loss':>10}  {'sigmasq':>9}  {'range_lon':>10}  "
          f"{'range_time':>11}  {'nugget':>8}  {'advec_lon':>10}")
    for _, row in sub.iterrows():
        print(
            f"  {int(row.n_obs_total):>8}  {row.loss:>10.2f}  "
            f"{row.est_sigmasq:>9.4f}  {row.est_range_lon:>10.4f}  "
            f"{row.est_range_time:>11.4f}  {row.est_nugget:>8.4f}  "
            f"{row.est_advec_lon:>10.4f}"
        )

print('\n' + '=' * 75)
print('SMOOTHNESS EFFECT AT FINEST RESOLUTION (x1)')
print('=' * 75)
x1 = df[df.R == 1].sort_values('fit_smooth')
print(f"  {'smooth':>7}  {'sigmasq':>9}  {'range_lon':>10}  {'range_time':>11}  "
      f"{'nugget':>8}  {'advec_lon':>10}")
for _, row in x1.iterrows():
    print(
        f"  {row.fit_smooth:>7}  {row.est_sigmasq:>9.4f}  {row.est_range_lon:>10.4f}  "
        f"{row.est_range_time:>11.4f}  {row.est_nugget:>8.4f}  {row.est_advec_lon:>10.4f}"
    )

In [ ]:
# ── parameter stability: range across resolutions ─────────────────────
print('PARAMETER RANGE ACROSS RESOLUTIONS (max - min per smooth)')
print(f"  {'param':12s}  {'smooth=0.5':>12}  {'smooth=1.5':>12}")
for k in P_LABELS:
    vals05 = df[df.fit_smooth == 0.5][f'est_{k}'].values
    vals15 = df[df.fit_smooth == 1.5][f'est_{k}'].values
    print(f"  {k:12s}  {vals05.max()-vals05.min():>12.4f}  {vals15.max()-vals15.min():>12.4f}")